In [ ]:
"""
=============================================================================
CHEQUEO DE CALIDAD Y ANÁLISIS EXPLORATORIO — DOS PANELES MAESTROS
(panel_maestro_ofrecen11 / panel_maestro_Solo_Primaria)
=============================================================================
Input:  Data/Processed/panel_maestro_ofrecen11.parquet
        Data/Processed/panel_maestro_Solo_Primaria.parquet
Output: Data/Processed/Figuras/diagnostico_visual_dos_paneles.pdf
        Data/Processed/Figuras/*.png
        Data/Processed/Figuras/resumen_calidad_dos_paneles.txt
Dependencies: pip install pandas numpy matplotlib pyarrow scipy statsmodels

Cada panel se analiza por separado (calidad, descriptivo, diagnóstico,
insights) y solo al final se cruzan en la sección de Cross-Panel Insights.

Outputs que quedan disponibles en el notebook:
    panel1_quality, panel2_quality, panel1_descriptive, panel2_descriptive,
    panel1_diagnostics, panel2_diagnostics, cross_panel_insights, key_insights
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

warnings.filterwarnings("ignore")

# =============================================================================
# 0. PATHS & CONFIG
# =============================================================================
ROOT = Path(r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data")
PROCESSED = ROOT / "Processed"
FIGURES = PROCESSED / "Figuras"
FIGURES.mkdir(parents=True, exist_ok=True)

PANEL_PATHS = {
    "ofrecen11": PROCESSED / "panel_maestro_ofrecen11.parquet",
    "solo_primaria": PROCESSED / "panel_maestro_Solo_Primaria.parquet",
}

TARGET = "TASA_DESERCION_MPIO"
ENTITY_COL = "SEDE_CODIGO"
YEAR_COL = "PERIODO_ANIO"

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

pdf = PdfPages(FIGURES / "diagnostico_visual_dos_paneles.pdf")
summary_lines = []


def log(msg):
    """Imprime en consola y guarda en el resumen de texto."""
    print(msg)
    summary_lines.append(msg)


def save_fig(fig, name):
    """Guarda la figura como PNG suelto y la agrega al PDF consolidado."""
    fig.tight_layout()
    fig.savefig(FIGURES / f"{name}.png", dpi=150, bbox_inches="tight")
    pdf.savefig(fig)
    plt.close(fig)


def load_panel(path, label):
    log(f"\n>>> CARGA — {label.upper()}")
    if not path.exists():
        raise FileNotFoundError(f"No se encontró: {path}")
    panel = pd.read_parquet(path)
    log(f"  Filas: {len(panel):,} | Columnas: {panel.shape[1]}")
    if YEAR_COL in panel.columns:
        log(f"  Años: {sorted(panel[YEAR_COL].dropna().unique().tolist())}")
    if ENTITY_COL in panel.columns:
        log(f"  Entidades únicas: {panel[ENTITY_COL].nunique():,}")
    if "SAMPLE" in panel.columns:
        log(f"  TRAIN/TEST: {panel['SAMPLE'].value_counts().to_dict()}")
    return panel


: 

In [ ]:
# =============================================================================
# 1. DATA QUALITY — funciones reutilizables (se aplican a cada panel por separado)
# =============================================================================
def data_quality_check(panel, label, target=TARGET, entity_col=ENTITY_COL, year_col=YEAR_COL):
    log(f"\n>>> DATA QUALITY — {label.upper()}")
    quality = {}

    quality["n_rows"] = len(panel)
    quality["n_cols"] = panel.shape[1]

    years = sorted(panel[year_col].dropna().unique().tolist()) if year_col in panel.columns else []
    quality["years"] = years
    quality["n_entities"] = panel[entity_col].nunique() if entity_col in panel.columns else None

    # --- Missing values ---
    pct_nulos = (panel.isnull().mean() * 100).sort_values(ascending=False)
    quality["pct_missing_overall"] = round(float(panel.isnull().mean().mean() * 100), 2)
    quality["vars_over_20pct_missing"] = pct_nulos[pct_nulos > 20].round(2).to_dict()
    quality["top_missing"] = pct_nulos.head(15).round(2).to_dict()

    top_nulos = pct_nulos.head(25)
    if len(top_nulos) > 0:
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.barh(top_nulos.index[::-1], top_nulos.values[::-1], color="firebrick")
        ax.set_xlabel("% de valores nulos")
        ax.set_title(f"[{label}] Top 25 variables con más nulos")
        ax.axvline(20, color="black", linestyle="--", linewidth=1, label="Umbral 20%")
        ax.legend()
        save_fig(fig, f"{label}_01_completitud")

    # --- Duplicados de llave ---
    if entity_col in panel.columns and year_col in panel.columns:
        dup_mask = panel.duplicated(subset=[entity_col, year_col], keep=False)
        quality["duplicate_keys"] = int(dup_mask.sum())
    else:
        quality["duplicate_keys"] = None

    # --- Panel balanceado / no balanceado ---
    if entity_col in panel.columns and year_col in panel.columns and years:
        years_per_entity = panel.groupby(entity_col)[year_col].nunique()
        quality["is_balanced"] = bool((years_per_entity == len(years)).all())
        quality["entities_with_all_years"] = int((years_per_entity == len(years)).sum())
        quality["entities_with_partial_years"] = int((years_per_entity < len(years)).sum())
        quality["min_years_per_entity"] = int(years_per_entity.min())
        quality["max_years_per_entity"] = int(years_per_entity.max())

    # --- Outliers/anomalías (regla IQR) sobre variables numéricas con cobertura suficiente ---
    num_cols = panel.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if panel[c].notna().sum() > 500]
    outlier_counts = {}
    for c in num_cols[:40]:
        s = panel[c].dropna()
        if len(s) < 30:
            continue
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            continue
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = int(((s < lower) | (s > upper)).sum())
        if n_out > 0:
            outlier_counts[c] = n_out
    quality["outlier_counts_top10"] = dict(
        sorted(outlier_counts.items(), key=lambda x: x[1], reverse=True)[:10]
    )

    # --- Cobertura del target ---
    if target in panel.columns:
        quality["target_coverage_pct"] = round(float(panel[target].notna().mean() * 100), 2)

    log(f"  Dimensiones: {quality['n_rows']:,} filas x {quality['n_cols']} columnas")
    log(f"  Entidades: {quality.get('n_entities')} | Años: {years}")
    log(f"  Missing promedio: {quality['pct_missing_overall']}% | "
        f"Variables con >20% nulos: {len(quality['vars_over_20pct_missing'])}")
    log(f"  Llaves duplicadas: {quality.get('duplicate_keys')}")
    log(f"  Panel balanceado: {quality.get('is_balanced')} "
        f"({quality.get('entities_with_all_years')} entidades completas de {quality.get('n_entities')})")
    log(f"  Variables con más outliers (IQR): {list(quality['outlier_counts_top10'].items())[:5]}")

    return quality


# =============================================================================
# 2. DESCRIPTIVE ANALYSIS — ¿qué pasó?
# =============================================================================
def descriptive_analysis(panel, label, target=TARGET, year_col=YEAR_COL, entity_col=ENTITY_COL):
    log(f"\n>>> DESCRIPTIVE ANALYSIS — {label.upper()}")
    descriptive = {}

    num_cols = panel.select_dtypes(include=[np.number]).columns.tolist()
    descriptive["summary_stats"] = panel[num_cols].describe().T

    # --- Tendencia temporal del target (obligatorio) ---
    if target in panel.columns and year_col in panel.columns:
        target_by_year = panel.groupby(year_col)[target].mean()
        descriptive["target_by_year"] = target_by_year

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(target_by_year.index, target_by_year.values, marker="o", color="darkred")
        ax.set_title(f"[{label}] {target} promedio por año")
        ax.set_xlabel("Año")
        ax.set_ylabel(target)
        if "FLAG_PANDEMIA" in panel.columns:
            ax.axvline(2020, color="gray", linestyle="--", alpha=0.6, label="Pandemia (2020)")
            ax.legend()
        save_fig(fig, f"{label}_02_tendencia_temporal")

        yoy = target_by_year.pct_change() * 100
        descriptive["target_yoy_growth_pct"] = yoy.round(2).dropna().to_dict()
        log(f"  Crecimiento interanual del target: {descriptive['target_yoy_growth_pct']}")

    # --- Variables con más variación temporal (CV de las medias anuales) ---
    if year_col in panel.columns:
        means_by_year = panel.groupby(year_col)[num_cols].mean()
        cv_year = (means_by_year.std() / means_by_year.mean().replace(0, np.nan)).abs()
        descriptive["vars_more_temporal_variation"] = (
            cv_year.dropna().sort_values(ascending=False).head(10).round(3).to_dict()
        )

    # --- Diferencias entre grupos ---
    group_diffs = {}
    if "REGION_DANE" in panel.columns and target in panel.columns:
        by_region = panel.groupby("REGION_DANE")[target].mean().sort_values(ascending=False)
        group_diffs["target_by_region"] = by_region.round(4).to_dict()

        regiones = panel["REGION_DANE"].dropna().unique().tolist()
        data_por_region = [panel.loc[panel["REGION_DANE"] == r, target].dropna() for r in regiones]
        if len(regiones) > 0:
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.boxplot(data_por_region)
            ax.set_xticklabels(regiones, rotation=30, ha="right")
            ax.set_title(f"[{label}] {target} por región DANE")
            ax.set_ylabel(target)
            save_fig(fig, f"{label}_03_target_por_region")

    for flag in ["FLAG_PDET", "FLAG_ZOMAC", "FLAG_PANDEMIA", "FLAG_DECLIVE_MATRICULA"]:
        if flag in panel.columns and target in panel.columns:
            g0 = panel.loc[panel[flag] == 0, target].mean()
            g1 = panel.loc[panel[flag] == 1, target].mean()
            if pd.notna(g0) and pd.notna(g1):
                group_diffs[flag] = {
                    "grupo_0": round(float(g0), 4),
                    "grupo_1": round(float(g1), 4),
                    "diferencia": round(float(g1 - g0), 4),
                }
    descriptive["group_differences"] = group_diffs

    # --- Rankings / slicing ---
    if entity_col in panel.columns and target in panel.columns:
        entity_avg = panel.groupby(entity_col)[target].mean()
        descriptive["top10_higher_risk_entities"] = (
            entity_avg.sort_values(ascending=False).head(10).round(4).to_dict()
        )
        descriptive["top10_lower_risk_entities"] = (
            entity_avg.sort_values(ascending=True).head(10).round(4).to_dict()
        )

    if "COD_DPTO_DANE" in panel.columns and target in panel.columns:
        dpto_avg = panel.groupby("COD_DPTO_DANE")[target].mean().sort_values(ascending=False)
        descriptive["ranking_departamentos"] = dpto_avg.head(10).round(4).to_dict()

    log(f"  Variables con mayor variación temporal: "
        f"{list(descriptive.get('vars_more_temporal_variation', {}).items())[:5]}")

    return descriptive


In [ ]:
# =============================================================================
# 3. DIAGNOSTIC ANALYSIS — ¿por qué podría estar pasando?
# =============================================================================
def diagnostic_analysis(panel, label, target=TARGET, entity_col=ENTITY_COL,
                         year_col=YEAR_COL, key_vars=None):
    log(f"\n>>> DIAGNOSTIC ANALYSIS — {label.upper()}")
    diagnostics = {}

    if key_vars is None:
        key_vars = [
            target, "TASA_REPITENCIA_MPIO", "DESERCION_LAG1", "REPITENCIA_LAG1",
            "DESERCION_MA2_LAG", "MATRICULA_TOTAL", "MATRICULA_PCT_CAMBIO",
            "INDICE_VULNERABILIDAD", "IPM_IPM", "ICFES_PROM_PUNT_GLOBAL",
            "FLAG_PDET", "FLAG_ZOMAC",
        ]
    key_vars = [c for c in key_vars if c in panel.columns and panel[c].notna().sum() > 500]

    # --- Correlación de Pearson con el target (universo amplio de variables) ---
    if target in panel.columns:
        num_cols = panel.select_dtypes(include=[np.number]).columns.tolist()
        num_cols = [c for c in num_cols if c != target and panel[c].notna().sum() > 500]

        corr_pearson = pd.Series(dtype=float)
        if num_cols:
            corr_pearson = (
                panel[num_cols + [target]].corr()[target]
                .drop(target, errors="ignore")
                .dropna()
                .sort_values(key=abs, ascending=False)
            )
        diagnostics["top_correlations_pearson"] = corr_pearson.head(15).round(4).to_dict()

        top15 = corr_pearson.head(15)
        if len(top15) > 0:
            fig, ax = plt.subplots(figsize=(10, 7))
            colors = ["firebrick" if v < 0 else "steelblue" for v in top15.values]
            ax.barh(top15.index[::-1], top15.values[::-1], color=colors[::-1])
            ax.axvline(0, color="black", linewidth=0.8)
            ax.set_xlabel("Correlación de Pearson")
            ax.set_title(f"[{label}] Top 15 correlaciones con {target}")
            save_fig(fig, f"{label}_04_top_correlaciones")

            # Spearman sobre las mismas variables — capta relaciones monotónicas no lineales
            common_vars = [c for c in top15.index if c in panel.columns]
            corr_spearman = (
                panel[common_vars + [target]].corr(method="spearman")[target]
                .drop(target, errors="ignore")
            )
            diagnostics["top_correlations_spearman"] = corr_spearman.round(4).to_dict()

            diffs = (corr_spearman - corr_pearson[common_vars]).abs().sort_values(ascending=False)
            diagnostics["pearson_vs_spearman_gap"] = diffs.head(5).round(4).to_dict()

    # --- Matriz de correlación — variables clave ---
    if len(key_vars) >= 2:
        corr_matrix = panel[key_vars].corr()
        diagnostics["correlation_matrix"] = corr_matrix

        fig, ax = plt.subplots(figsize=(9, 8))
        im = ax.imshow(corr_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1)
        ax.set_xticks(range(len(key_vars)))
        ax.set_yticks(range(len(key_vars)))
        ax.set_xticklabels(key_vars, rotation=90)
        ax.set_yticklabels(key_vars)
        for i in range(len(key_vars)):
            for j in range(len(key_vars)):
                val = corr_matrix.values[i, j]
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if abs(val) > 0.5 else "black")
        fig.colorbar(im, ax=ax, shrink=0.8, label="Correlación")
        ax.set_title(f"[{label}] Matriz de correlación — variables clave")
        save_fig(fig, f"{label}_05_matriz_correlacion")

    # --- VIF ---
    # NORMALIZACIÓN: se estandarizan (z-score) las variables SOLO para este cálculo
    # de VIF, porque el estadístico es sensible a la escala relativa entre predictores.
    # No se modifica el panel original ni ninguna otra salida de este notebook.
    vif_vars = [c for c in key_vars if c != target]
    if len(vif_vars) >= 2:
        vif_data = panel[vif_vars].dropna()
        if len(vif_data) > 100:
            vif_std = (vif_data - vif_data.mean()) / vif_data.std(ddof=0)
            vif_std = vif_std.replace([np.inf, -np.inf], np.nan).dropna()
            vif_std = vif_std.loc[:, vif_std.std(ddof=0) > 0]  # descarta constantes
            try:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                X = vif_std.values
                vif_scores = {
                    vif_std.columns[i]: round(float(variance_inflation_factor(X, i)), 2)
                    for i in range(X.shape[1])
                }
                diagnostics["vif"] = dict(sorted(vif_scores.items(), key=lambda x: x[1], reverse=True))
                diagnostics["vif_high"] = {k: v for k, v in diagnostics["vif"].items() if v > 5}
                log(f"  [Normalización aplicada] z-score sobre {list(vif_std.columns)} "
                    f"únicamente para el cálculo de VIF.")

                vif_sorted = pd.Series(diagnostics["vif"]).sort_values()
                fig, ax = plt.subplots(figsize=(8, 6))
                colors = ["firebrick" if v > 5 else "steelblue" for v in vif_sorted.values]
                ax.barh(vif_sorted.index, vif_sorted.values, color=colors)
                ax.axvline(5, color="black", linestyle="--", label="Umbral VIF=5")
                ax.set_xlabel("VIF")
                ax.set_title(f"[{label}] Factor de inflación de varianza (variables estandarizadas)")
                ax.legend()
                save_fig(fig, f"{label}_06_vif")
            except Exception as e:
                log(f"  [WARN] No se pudo calcular VIF: {e}")
                diagnostics["vif"] = {}
                diagnostics["vif_high"] = {}

    # --- Lag relationships ---
    lag_pairs = [
        ("DESERCION_LAG1", target),
        ("DESERCION_MA2_LAG", target),
        ("REPITENCIA_LAG1", "TASA_REPITENCIA_MPIO"),
    ]
    lag_results = {}
    for lag_var, tgt_var in lag_pairs:
        if lag_var in panel.columns and tgt_var in panel.columns:
            lag_data = panel[[tgt_var, lag_var]].dropna()
            if len(lag_data) > 30:
                r = lag_data.corr().loc[tgt_var, lag_var]
                lag_results[f"{lag_var}_vs_{tgt_var}"] = {"r": round(float(r), 4), "r2": round(float(r ** 2), 4)}
    diagnostics["lag_relationships"] = lag_results

    if "DESERCION_LAG1" in panel.columns and target in panel.columns:
        lag_data = panel[[target, "DESERCION_LAG1"]].dropna()
        if len(lag_data) > 0:
            r = lag_data.corr().loc[target, "DESERCION_LAG1"]
            fig, ax = plt.subplots(figsize=(6.5, 6.5))
            ax.scatter(lag_data["DESERCION_LAG1"], lag_data[target], alpha=0.15, s=8, color="steelblue")
            lims = [lag_data.min().min(), lag_data.max().max()]
            ax.plot(lims, lims, color="black", linestyle="--", linewidth=1, label="y = x")
            ax.set_xlabel("DESERCION_LAG1 (año anterior)")
            ax.set_ylabel(target)
            ax.set_title(f"[{label}] Autocorrelación lag-1 (R={r:.3f}, R²={r ** 2:.3f})")
            ax.legend()
            save_fig(fig, f"{label}_07_autocorrelacion_lag1")

    # --- Within vs between variation (descomposición panel) ---
    if entity_col in panel.columns and target in panel.columns:
        valid = panel[[entity_col, target]].dropna()
        if len(valid) > 0:
            entity_means_full = valid.groupby(entity_col)[target].transform("mean")
            between_var = valid.groupby(entity_col)[target].mean().var()
            within_var = (valid[target] - entity_means_full).var()
            total_var = valid[target].var()
            diagnostics["within_between_variation"] = {
                "between_var": round(float(between_var), 6) if pd.notna(between_var) else None,
                "within_var": round(float(within_var), 6) if pd.notna(within_var) else None,
                "total_var": round(float(total_var), 6) if pd.notna(total_var) else None,
                "pct_between": round(float(between_var / total_var) * 100, 1) if total_var else None,
                "pct_within": round(float(within_var / total_var) * 100, 1) if total_var else None,
            }
            log(f"  Variación between (entre sedes): {diagnostics['within_between_variation']['pct_between']}% | "
                f"Variación within (dentro de cada sede): {diagnostics['within_between_variation']['pct_within']}%")

    # --- Outliers y posibles explicaciones ---
    if target in panel.columns:
        target_data = panel[target].dropna()
        if len(target_data) > 0:
            q1, q3 = target_data.quantile(0.25), target_data.quantile(0.75)
            iqr = q3 - q1
            lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            outlier_mask = (panel[target] < lower) | (panel[target] > upper)
            diagnostics["target_outliers_n"] = int(outlier_mask.sum())
            diagnostics["target_outliers_pct"] = round(float(outlier_mask.mean() * 100), 2)

            outlier_explain = {}
            for flag in ["FLAG_PDET", "FLAG_ZOMAC", "FLAG_PANDEMIA"]:
                if flag in panel.columns:
                    pct_in_outliers = panel.loc[outlier_mask, flag].mean() * 100
                    pct_in_general = panel[flag].mean() * 100
                    if pd.notna(pct_in_outliers) and pd.notna(pct_in_general):
                        outlier_explain[flag] = {
                            "pct_en_outliers": round(float(pct_in_outliers), 1),
                            "pct_general": round(float(pct_in_general), 1),
                        }
            diagnostics["outlier_explanations"] = outlier_explain

            fig, ax = plt.subplots(figsize=(6, 6))
            ax.boxplot(target_data)
            ax.set_xticklabels([target])
            ax.set_title(f"[{label}] Outliers en {target} (regla IQR)")
            save_fig(fig, f"{label}_08_outliers_target")

    return diagnostics


In [ ]:
# =============================================================================
# 4. INSIGHT DISCOVERY
# Para cada insight: qué pasó -> evidencia -> posible explicación -> por qué
# importa -> qué investigar después. Correlación no implica causalidad.
# =============================================================================
def discover_insights(label, quality, descriptive, diagnostics, target=TARGET):
    log(f"\n>>> INSIGHT DISCOVERY — {label.upper()}")
    insights = []

    def add_insight(que_paso, evidencia, explicacion, importancia, siguiente_paso):
        insights.append({
            "panel": label,
            "que_paso": que_paso,
            "evidencia": evidencia,
            "posible_explicacion": explicacion,
            "por_que_importa": importancia,
            "que_investigar_despues": siguiente_paso,
        })

    top_corr = diagnostics.get("top_correlations_pearson", {})
    if top_corr:
        var, val = list(top_corr.items())[0]
        add_insight(
            f"{var} muestra la correlación más fuerte con {target} (r={val}).",
            f"Correlación de Pearson = {val}, calculada sobre variables con cobertura suficiente.",
            "Posible relación estructural o de persistencia temporal del fenómeno; "
            "no implica causalidad directa.",
            "Candidata prioritaria para el modelo predictivo baseline.",
            "Evaluar su poder predictivo fuera de muestra y su estabilidad temporal (train vs test).",
        )

    vif_high = diagnostics.get("vif_high", {})
    if vif_high:
        add_insight(
            f"{len(vif_high)} variable(s) presentan VIF > 5: {list(vif_high.keys())}.",
            f"VIF calculado sobre variables estandarizadas: {vif_high}.",
            "Multicolinealidad entre predictores, probablemente por construcción compartida "
            "(lags derivados de la misma tasa municipal).",
            "Puede inflar errores estándar y distorsionar coeficientes en modelos lineales.",
            "Evaluar remoción, combinación (PCA) o modelos robustos a colinealidad "
            "(árboles, regularización L1/L2).",
        )

    yoy = descriptive.get("target_yoy_growth_pct", {})
    if yoy:
        year_max_change = max(yoy, key=lambda k: abs(yoy[k]))
        add_insight(
            f"El mayor cambio interanual del target ocurre en {year_max_change} "
            f"({yoy[year_max_change]}%).",
            f"Serie de crecimiento interanual: {yoy}.",
            "Coincide con choques conocidos del sistema educativo colombiano (p. ej. pandemia "
            "2020), asociados a cierres de sede y migración a educación remota.",
            "Un choque estructural de este tipo puede requerir tratamiento especial en el "
            "modelo (dummy de año, ventana de entrenamiento separada).",
            "Probar el modelo con y sin el año de choque para medir sensibilidad.",
        )

    top_risk = descriptive.get("top10_higher_risk_entities", {})
    if top_risk:
        add_insight(
            "Existe un grupo reducido de sedes con tasas de deserción muy por encima "
            "del promedio del panel.",
            f"Top sedes con mayor tasa promedio: {list(top_risk.items())[:3]}.",
            "Concentración geográfica o socioeconómica de vulnerabilidad "
            "(zonas PDET/ZOMAC, dispersión rural).",
            "Estas sedes son las de mayor valor de negocio para el sistema de alerta: "
            "priorizarlas mejora el retorno de la intervención.",
            "Cruzar esta lista con FLAG_PDET/FLAG_ZOMAC y con la región DANE para confirmar "
            "el patrón geográfico.",
        )

    group_diffs = descriptive.get("group_differences", {})
    flag_diffs = {k: v["diferencia"] for k, v in group_diffs.items() if isinstance(v, dict) and "diferencia" in v}
    if flag_diffs:
        flag_max = max(flag_diffs, key=lambda k: abs(flag_diffs[k]))
        add_insight(
            f"{flag_max} está asociado a la mayor diferencia en {target} entre grupos "
            f"({flag_diffs[flag_max]}).",
            f"Diferencias por flag: {flag_diffs}.",
            "El flag captura una condición estructural (conflicto, pobreza o dispersión, "
            "según el caso) que coincide con mayor riesgo de deserción.",
            "Sugiere una variable de alto valor discriminante para el modelo y para la "
            "focalización de política pública.",
            "Verificar si el efecto se mantiene al controlar por región y año.",
        )

    lag_rel = diagnostics.get("lag_relationships", {})
    if lag_rel:
        best_lag = max(lag_rel, key=lambda k: lag_rel[k]["r2"])
        add_insight(
            f"La relación lagged más fuerte es {best_lag} (R²={lag_rel[best_lag]['r2']}).",
            f"Relaciones lag evaluadas: {lag_rel}.",
            "Persistencia temporal del fenómeno de deserción/repitencia a nivel municipal, "
            "esperable en indicadores sociales con inercia estructural.",
            "Confirma que las variables lag están entre las más informativas para el "
            "modelo baseline.",
            "Evaluar si agregar lags de mayor orden (t-2, t-3) mejora el poder predictivo "
            "sin perder demasiadas observaciones.",
        )

    wb = diagnostics.get("within_between_variation", {})
    if wb and wb.get("pct_between") is not None:
        dominant = "between" if wb["pct_between"] > wb["pct_within"] else "within"
        add_insight(
            f"La variación {dominant} explica la mayor parte de la varianza del target "
            f"(between={wb['pct_between']}%, within={wb['pct_within']}%).",
            f"Descomposición de varianza: {wb}.",
            "Si domina la variación between, el riesgo es principalmente una característica "
            "estructural de la sede/municipio; si domina within, el riesgo cambia "
            "sustancialmente año a año dentro de la misma sede.",
            "Determina si conviene un modelo con efectos fijos por sede/municipio o uno que "
            "capture dinámica temporal de forma más agresiva.",
            "Probar un modelo con efectos fijos de entidad como benchmark adicional al "
            "baseline de regresión logística.",
        )

    if diagnostics.get("target_outliers_pct"):
        exp = diagnostics.get("outlier_explanations", {})
        add_insight(
            f"{diagnostics['target_outliers_pct']}% de las observaciones del target son "
            f"outliers según la regla IQR.",
            f"N outliers = {diagnostics.get('target_outliers_n')}. Prevalencia de flags dentro "
            f"de outliers vs. general: {exp}.",
            "Los outliers pueden corresponder a sedes muy pequeñas donde un cambio de pocos "
            "estudiantes genera tasas extremas, o a choques puntuales (desplazamiento, "
            "cierre parcial).",
            "Outliers no tratados pueden distorsionar los coeficientes de un modelo lineal y "
            "sesgar el umbral de clasificación.",
            "Revisar si los outliers se concentran en sedes de matrícula muy baja y evaluar "
            "un filtro de tamaño mínimo o un tratamiento robusto.",
        )

    for ins in insights:
        log(f"  • {ins['que_paso']}")

    return insights


# =============================================================================
# 5. CROSS-PANEL INSIGHTS — solo después de analizar cada panel por separado
# =============================================================================
def build_cross_panel_insights(p1_label, p1_quality, p1_descriptive, p1_diagnostics,
                                p2_label, p2_quality, p2_descriptive, p2_diagnostics,
                                target=TARGET):
    log(f"\n>>> CROSS-PANEL INSIGHTS — {p1_label.upper()} vs {p2_label.upper()}")
    cross = []

    if p1_quality.get("n_entities") and p2_quality.get("n_entities"):
        cross.append({
            "que_paso": f"El panel {p1_label} tiene {p1_quality['n_entities']:,} sedes y el "
                        f"panel {p2_label} tiene {p2_quality['n_entities']:,}.",
            "evidencia": f"{p1_label}: {p1_quality['n_rows']:,} filas | "
                         f"{p2_label}: {p2_quality['n_rows']:,} filas.",
            "posible_explicacion": "La mayoría de las sedes del sistema público colombiano son "
                                    "de solo primaria/básica y no ofrecen el ciclo de media completo.",
            "por_que_importa": "El modelo debe considerar el desbalance de tamaño entre ambos "
                                "segmentos al definir estrategias de entrenamiento y evaluación.",
            "que_investigar_despues": "Evaluar si conviene un modelo único con TIENE_GRADO_11 "
                                       "como feature, o dos modelos especializados.",
        })

    t1 = p1_descriptive.get("target_by_year")
    t2 = p2_descriptive.get("target_by_year")
    if t1 is not None and t2 is not None and len(t1) > 0 and len(t2) > 0:
        mean1, mean2 = float(t1.mean()), float(t2.mean())
        cross.append({
            "que_paso": f"El nivel promedio de {target} es "
                        f"{'mayor' if mean1 > mean2 else 'menor'} en {p1_label} ({mean1:.4f}) "
                        f"que en {p2_label} ({mean2:.4f}).",
            "evidencia": f"{p1_label} promedio={mean1:.4f} | {p2_label} promedio={mean2:.4f}.",
            "posible_explicacion": "Las sedes con oferta de media suelen ubicarse en cabeceras "
                                    "municipales con mejor infraestructura, mientras que las de "
                                    "solo primaria son con frecuencia rurales dispersas.",
            "por_que_importa": "Confirma que los dos segmentos tienen perfiles de riesgo "
                                "distintos y refuerza la decisión de analizarlos por separado "
                                "antes de combinarlos.",
            "que_investigar_despues": "Verificar si la diferencia persiste al controlar por "
                                       "región y ruralidad.",
        })

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(t1.index, t1.values, marker="o", label=p1_label, color="steelblue")
        ax.plot(t2.index, t2.values, marker="o", label=p2_label, color="firebrick")
        ax.set_title(f"{target}: {p1_label} vs {p2_label}")
        ax.set_xlabel("Año")
        ax.set_ylabel(target)
        ax.legend()
        save_fig(fig, "cross_01_tendencia_comparada")

    c1 = p1_diagnostics.get("top_correlations_pearson", {})
    c2 = p2_diagnostics.get("top_correlations_pearson", {})
    common_vars = set(c1.keys()) & set(c2.keys())
    if common_vars:
        diffs = {v: round(c1[v] - c2[v], 4) for v in common_vars}
        var_max_diff = max(diffs, key=lambda k: abs(diffs[k]))
        cross.append({
            "que_paso": f"La variable {var_max_diff} tiene una relación con el target "
                        f"notablemente distinta entre paneles (diferencia={diffs[var_max_diff]}).",
            "evidencia": f"{p1_label}: r={c1[var_max_diff]} | {p2_label}: r={c2[var_max_diff]}. "
                         f"Diferencias completas: {diffs}.",
            "posible_explicacion": "El mecanismo detrás de esa variable puede operar distinto "
                                    "según el nivel educativo ofrecido en la sede.",
            "por_que_importa": "Sugiere que un modelo único con coeficientes fijos podría no "
                                "capturar bien el fenómeno en ambos segmentos.",
            "que_investigar_despues": "Probar una interacción entre esa variable y "
                                       "TIENE_GRADO_11, o modelos separados por segmento.",
        })

    if p1_quality.get("pct_missing_overall") is not None and p2_quality.get("pct_missing_overall") is not None:
        cross.append({
            "que_paso": f"La cobertura de missing difiere entre paneles ({p1_label}: "
                        f"{p1_quality['pct_missing_overall']}% | {p2_label}: "
                        f"{p2_quality['pct_missing_overall']}%).",
            "evidencia": f"Variables con >20% de nulos — {p1_label}: "
                         f"{len(p1_quality['vars_over_20pct_missing'])} | {p2_label}: "
                         f"{len(p2_quality['vars_over_20pct_missing'])}.",
            "posible_explicacion": "El panel Solo_Primaria carece por definición de variables "
                                    "ICFES, lo que eleva su tasa de missing en ese bloque.",
            "por_que_importa": "El tratamiento de missing (imputación, exclusión, modelos que "
                                "toleren NaN) debe definirse por separado para cada panel.",
            "que_investigar_despues": "Confirmar que el missing de variables ICFES en "
                                       "Solo_Primaria es 100% por diseño y no por error de carga.",
        })

    for c in cross:
        log(f"  • {c['que_paso']}")

    return cross


In [ ]:
# =============================================================================
# 6. EJECUCIÓN — PANEL 1: OFRECEN11 (analizado primero, en solitario)
# =============================================================================
panel1 = load_panel(PANEL_PATHS["ofrecen11"], "ofrecen11")

panel1_quality = data_quality_check(panel1, "ofrecen11")
panel1_descriptive = descriptive_analysis(panel1, "ofrecen11")
panel1_diagnostics = diagnostic_analysis(panel1, "ofrecen11")
panel1_insights = discover_insights("ofrecen11", panel1_quality, panel1_descriptive, panel1_diagnostics)


In [ ]:
# =============================================================================
# 7. EJECUCIÓN — PANEL 2: SOLO_PRIMARIA (analizado por separado)
# =============================================================================
panel2 = load_panel(PANEL_PATHS["solo_primaria"], "solo_primaria")

panel2_quality = data_quality_check(panel2, "solo_primaria")
panel2_descriptive = descriptive_analysis(panel2, "solo_primaria")
panel2_diagnostics = diagnostic_analysis(panel2, "solo_primaria")
panel2_insights = discover_insights("solo_primaria", panel2_quality, panel2_descriptive, panel2_diagnostics)


In [ ]:
# =============================================================================
# 8. CROSS-PANEL INSIGHTS Y CONSOLIDACIÓN DE KEY INSIGHTS
# Se ejecuta únicamente después de tener ambos análisis individuales completos.
# =============================================================================
cross_panel_insights = build_cross_panel_insights(
    "ofrecen11", panel1_quality, panel1_descriptive, panel1_diagnostics,
    "solo_primaria", panel2_quality, panel2_descriptive, panel2_diagnostics,
)

key_insights = panel1_insights + panel2_insights + cross_panel_insights

log(f"\n>>> KEY INSIGHTS CONSOLIDADOS: {len(key_insights)} "
    f"({len(panel1_insights)} de ofrecen11, {len(panel2_insights)} de solo_primaria, "
    f"{len(cross_panel_insights)} cross-panel)")


In [ ]:
# =============================================================================
# 9. CIERRE
# =============================================================================
pdf.close()

resumen_path = FIGURES / "resumen_calidad_dos_paneles.txt"
resumen_path.write_text("\n".join(summary_lines), encoding="utf-8")

log("\n>>> ANÁLISIS DE DOS PANELES COMPLETO")
log(f"  PDF:     {FIGURES / 'diagnostico_visual_dos_paneles.pdf'}")
log(f"  PNGs:    {FIGURES}")
log(f"  Resumen: {resumen_path}")
log("  Outputs disponibles: panel1_quality, panel2_quality, panel1_descriptive, "
    "panel2_descriptive, panel1_diagnostics, panel2_diagnostics, cross_panel_insights, key_insights")
